<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 12. Streamlit, Docker, финальная сборка — подробная методичка

Эта методичка продолжает Этапы 1–11. Предполагается, что у вас:
- `docker compose ps` — `db`, `redis`, `qdrant`, `ollama`, `app`, `worker`, `beat` в `Up`/`healthy`;
- `/health` — 200, все `services.*.healthy: true`;
- `/translate`, `/hitl/uncertain`, `/hitl/uncertain/{id}/confirm` — работают;
- HITL → Celery → переиндексация работает (Этап 11);
- `backend/api/main.py` содержит `health`, `auth`, `translate`, `admin`, `hitl` роутеры.

Если чего-то нет — вернитесь к Этапу 11.

**Цель этапа:** Streamlit UI (8 страниц), Nginx в `docker-compose.yml`, README, end-to-end тест.

**Недель:** 1 + буфер.

**Что вы получите в конце этапа:**

- `streamlit_app/app.py` — главная с логином/регистрацией, быстрым переводом, ссылками на 8 страниц.
- `streamlit_app/pages/1-8` — 8 страниц.
- `streamlit_app/utils/` — 4 файла (`api_client.py`, `auth_guard.py`, `helpers.py`, `languages.py`).
- `streamlit_app/requirements.txt`.
- `infrastructure/docker-compose.yml` — сервис `nginx`.
- `infrastructure/nginx.conf` — прокси, gzip, security headers.
- `README.md` — полный.
- `tests/integration/test_end_to_end.py`.
- (Опционально, бонус) — eval-скрипты `run_eval.py`, `mode_labels.py`, `paired_bootstrap.py`, `report_tables.py`, `check_tm_leakage.py`, `human_eval/`.

**Критерии готовности:**

- [ ] `docker compose ps` — `nginx` в `Up`, все сервисы `healthy`.
- [ ] `curl http://localhost/health` (через nginx) — 200.
- [ ] `streamlit run streamlit_app/app.py --server.port 8501` — открывается на `:8501`.
- [ ] Логин/регистрация работают через UI.
- [ ] `/translate` через UI возвращает перевод.
- [ ] HITL: очередь видна, confirm/reject работают.
- [ ] `pytest tests/integration/test_end_to_end.py -v` — зелёный.
- [ ] `README.md` содержит quick start, архитектуру, данные, ограничения, инструкцию для Streamlit.
- [ ] Ветка `week-12` создана, коммит сделан.

> **Важно.** Streamlit **не в `docker-compose.yml`**. Запускается локально на хосте (`streamlit run streamlit_app/app.py --server.port 8501`). Причина: образ Streamlit — ~1 ГБ ради UI, а production-образ `app` уже ~700 МБ (torch/transformers). Смешивать не нужно. Nginx проксирует только `app:8000` (API). Streamlit — отдельный dev-инструмент, обращается к API через `http://localhost:8000`.

---

## 12.0. Подготовка: ветка и зависимости

```powershell
cd D:\RUNG
git checkout -b week-12
```

Установите Streamlit локально (не через Poetry — Streamlit — dev-инструмент, не часть бэкенда):

```powershell
python -m pip install streamlit requests
```

Или через `requirements.txt` (12.14).

Проверьте, что в `backend/core/constants.py` (Этап 2) есть `LANGUAGES` с 18 языками — он **не** импортируется в Streamlit напрямую (Streamlit не имеет доступа к `backend/`), но копия должна совпадать.

---

## 12.1. `streamlit_app/utils/languages.py`

**Что это:** словарь `LANGUAGES` (код → название), синхронизированный с `backend/core/constants.py`.

**Зачем это нужно:** Streamlit не импортирует `backend/` (запускается отдельно на хосте, backend — в контейнере). `LANGUAGES` копируется вручную. Если добавите 19-й язык в `backend/core/constants.py` — не забудьте и здесь.

Создайте `streamlit_app/utils/languages.py`:

```python
# streamlit_app/utils/languages.py
LANGUAGES = {
    "en": "English",
    "ru": "Russian",
    "tg": "Tajik",
    "tt": "Tatar",
    "fa": "Persian",
    "os": "Ossetian",
    "ar": "Arabic",
    "av": "Avar",
    "ba": "Bashkir",
    "bg": "Bulgarian",
    "id": "Indonesian",
    "kk": "Kazakh",
    "ky": "Kyrgyz",
    "ms": "Malay",
    "tk": "Turkmen",
    "tr": "Turkish",
    "uz": "Uzbek",
    "udm": "Udmurt",
}
```

### Разбор

**Почему `dict`, а не `list`.** Streamlit `st.selectbox(options=list(LANGUAGES.keys()), format_func=lambda x: LANGUAGES[x])` — даёт `(code, name)` пары. Валидация и lookup — O(1).

**Почему 18, а не 14.** `LANGUAGES` — полный справочник для UI. `KNOWN_LANGUAGES` (14) — только для Ollama-фильтра (Ensemble). Все 18 доступны для `/translate` — для `udm`/`os`/`av`/`tk` `translator_manager` выберет HF.

**Почему порядок важен.** Streamlit `selectbox` показывает в порядке вставки. `en` первым, `ru` вторым — для удобства (типичные пары).

---

## 12.2. `streamlit_app/utils/api_client.py`

**Что это:** обёртка над `requests` для API. Обрабатывает 401 (сброс `session_state`), 503 от `/health` (валидный ответ), Pydantic-ошибки `{detail: [{loc, msg}]}`.

**Зачем это нужно:** каждая Streamlit-страница делает 2–10 HTTP-запросов. Без общего клиента — дублирование (headers, обработка ошибок, `_clear_session_state`). Один клиент — одна точка правды.

Создайте `streamlit_app/utils/api_client.py`:

```python
# streamlit_app/utils/api_client.py
import requests
import streamlit as st
from typing import Optional, Dict, Any, List
from urllib.parse import quote

try:
    API_URL = st.secrets.get("API_URL", "http://localhost:8000")
except Exception:
    API_URL = "http://localhost:8000"

_EPHEMERAL_SUFFIXES = ("_result", "_data", "_key")


def get_headers() -> Dict[str, str]:
    token = st.session_state.get("access_token")
    if token:
        return {"Authorization": f"Bearer {token}"}
    return {}


def _extract_detail(response: requests.Response) -> str:
    try:
        payload = response.json()
    except ValueError:
        return (response.text or "")[:500]

    detail = payload.get("detail") if isinstance(payload, dict) else None

    if isinstance(detail, str):
        return detail

    if isinstance(detail, list):
        parts = []
        for item in detail:
            if isinstance(item, dict):
                loc = ".".join(str(x) for x in item.get("loc", []))
                msg = item.get("msg", "")
                parts.append(f"{loc}: {msg}" if loc else msg)
            else:
                parts.append(str(item))
        return "; ".join(p for p in parts if p)

    return ""


def api_request(method: str, endpoint: str, **kwargs) -> Optional[Dict[str, Any]]:
    url = f"{API_URL}{endpoint}"
    headers = get_headers()
    if "headers" in kwargs:
        headers.update(kwargs.pop("headers"))
    if "files" not in kwargs:
        headers.setdefault("Content-Type", "application/json")

    try:
        response = requests.request(method, url, headers=headers, **kwargs)

        if endpoint == "/health" and response.status_code == 503:
            try:
                return response.json()
            except ValueError:
                return {}

        response.raise_for_status()
        return response.json() if response.content else {}

    except requests.exceptions.RequestException as e:
        status = e.response.status_code if e.response is not None else "??"
        detail = _extract_detail(e.response) if e.response is not None else ""
        message = detail or str(e)
        st.error(f"API Error ({status}): {message}")

        if e.response is not None and e.response.status_code == 401:
            _clear_session_state()

        return None


def _clear_session_state() -> None:
    st.session_state.access_token = None
    st.session_state.user = None
    for key in list(st.session_state.keys()):
        if isinstance(key, str) and key.endswith(_EPHEMERAL_SUFFIXES):
            del st.session_state[key]


def login(email: str, password: str) -> bool:
    result = api_request(
        "POST", "/auth/login",
        json={"email": email, "password": password},
    )
    if result and "access_token" in result:
        st.session_state.access_token = result["access_token"]
        st.session_state.user = result
        return True
    return False


def register(email: str, username: str, password: str, full_name: str = "") -> bool:
    result = api_request("POST", "/auth/register", json={
        "email": email,
        "username": username,
        "password": password,
        "full_name": full_name,
    })
    if result and "access_token" in result:
        st.session_state.access_token = result["access_token"]
        st.session_state.user = result
        return True
    return False


def logout():
    _clear_session_state()


def get_me() -> Optional[Dict]:
    return api_request("GET", "/auth/me")


def translate(
    text: str,
    source_lang: str = "en",
    target_lang: str = "ru",
    engine: str = "auto",
) -> Optional[Dict]:
    return api_request("POST", "/translate", json={
        "text": text,
        "source_lang": source_lang,
        "target_lang": target_lang,
        "engine": engine,
    })


def translate_multi(
    text: str,
    source_lang: str,
    target_langs: List[str],
    engine: str = "auto",
) -> Optional[Dict]:
    return api_request("POST", "/translate/multi", json={
        "text": text,
        "source_lang": source_lang,
        "target_langs": target_langs,
        "engine": engine,
    })


def translate_batch(
    texts: List[str],
    source_lang: str,
    target_lang: str,
    engine: str = "auto",
) -> Optional[Dict]:
    return api_request("POST", "/translate/batch", json={
        "texts": texts,
        "source_lang": source_lang,
        "target_lang": target_lang,
        "engine": engine,
    })


def translate_ensemble(
    text: str,
    source_lang: str,
    target_lang: str,
    models: List[str],
) -> Optional[Dict]:
    return api_request("POST", "/translate/ensemble", json={
        "text": text,
        "source_lang": source_lang,
        "target_lang": target_lang,
        "models": models,
    })


def add_glossary(en: str, tgt: str, source_lang: str = "en", target_lang: str = "ru") -> Optional[Dict]:
    return api_request("POST", "/admin/glossary/add", json={
        "en": en, "tgt": tgt,
        "source_lang": source_lang, "target_lang": target_lang,
    })


def list_glossary(source_lang: str = None, target_lang: str = None, limit: int = 100, offset: int = 0) -> Optional[Dict]:
    params = {"limit": limit, "offset": offset}
    if source_lang: params["source_lang"] = source_lang
    if target_lang: params["target_lang"] = target_lang
    return api_request("GET", "/admin/glossary/list", params=params)


def import_glossary_csv(file) -> Optional[Dict]:
    return api_request("POST", "/admin/glossary/import-csv", files={"file": file})


def delete_glossary(entry_id: str) -> Optional[Dict]:
    return api_request("DELETE", f"/admin/glossary/delete/{quote(entry_id, safe='')}")


def add_tm(en: str, tgt: str, source_lang: str = "en", target_lang: str = "ru") -> Optional[Dict]:
    return api_request("POST", "/admin/tm/add", json={
        "en": en, "tgt": tgt,
        "source_lang": source_lang, "target_lang": target_lang,
    })


def list_tm(source_lang: str = None, target_lang: str = None, limit: int = 100, offset: int = 0) -> Optional[Dict]:
    params = {"limit": limit, "offset": offset}
    if source_lang: params["source_lang"] = source_lang
    if target_lang: params["target_lang"] = target_lang
    return api_request("GET", "/admin/tm/list", params=params)


def import_tm_csv(file) -> Optional[Dict]:
    return api_request("POST", "/admin/tm/import-csv", files={"file": file})


def delete_tm(entry_id: str) -> Optional[Dict]:
    return api_request("DELETE", f"/admin/tm/delete/{quote(entry_id, safe='')}")


def clear_cache(source_lang: str = None, target_lang: str = None, eval_mode: str = None) -> Optional[Dict]:
    params = {}
    if source_lang: params["source_lang"] = source_lang
    if target_lang: params["target_lang"] = target_lang
    if eval_mode: params["eval_mode"] = eval_mode
    return api_request("DELETE", "/admin/cache/clear", params=params)


def get_cache_stats() -> Optional[Dict]:
    return api_request("GET", "/admin/cache/stats")


def save_correction(
    source_text: str, source_lang: str, target_lang: str,
    corrected_translation: str, original_translation: str = "",
) -> Optional[Dict]:
    return api_request("POST", "/hitl/save-correction", json={
        "source_text": source_text,
        "source_lang": source_lang,
        "target_lang": target_lang,
        "corrected_translation": corrected_translation,
        "original_translation": original_translation,
    })


def get_uncertain(source_lang: str = None, target_lang: str = None, limit: int = 10, offset: int = 0) -> Optional[Dict]:
    params = {"limit": limit, "offset": offset}
    if source_lang: params["source_lang"] = source_lang
    if target_lang: params["target_lang"] = target_lang
    return api_request("GET", "/hitl/uncertain", params=params)


def confirm_uncertain(example_id: str, corrected_translation: str) -> Optional[Dict]:
    return api_request(
        "POST", f"/hitl/uncertain/{quote(example_id, safe='')}/confirm",
        json={"corrected_translation": corrected_translation},
    )


def reject_uncertain(example_id: str) -> Optional[Dict]:
    return api_request(
        "POST", f"/hitl/uncertain/{quote(example_id, safe='')}/reject",
    )


def get_uncertain_stats() -> Optional[Dict]:
    return api_request("GET", "/hitl/uncertain/stats")


def run_evaluation(sources: List[str] = None, hypotheses: List[str] = None, references: List[str] = None) -> Optional[Dict]:
    payload = {"hypotheses": hypotheses, "references": references}
    if sources:
        payload["sources"] = sources
    return api_request("POST", "/eval/run", json=payload)


def get_health() -> Optional[Dict]:
    return api_request("GET", "/health")


def get_healthz() -> Optional[Dict]:
    return api_request("GET", "/healthz")
```

### Разбор ключевых решений

**Почему `API_URL` из `st.secrets.get("API_URL", ...)`, а не `os.getenv`.** Streamlit читает секреты из `.streamlit/secrets.toml` (не `.env`). `st.secrets.get` — правильный API. Fallback `"http://localhost:8000"` — для локального запуска без secrets.

**Почему `try/except` вокруг `st.secrets.get`.** Если `secrets.toml` нет и `API_URL` не задан — `st.secrets.get` может бросить `StreamlitSecretNotFoundError`. `try/except` → fallback на localhost.

**Почему `_EPHEMERAL_SUFFIXES = ("_result", "_data", "_key")`.** Ключи `session_state`, привязанные к одному пользователю (`quick_result`, `glossary_data`, `glossary_filter_key`). При logout/401 — удаляются, чтобы данные предыдущего юзера не «протекали» в новую сессию. Удаление через `del st.session_state[key]` — работает для обоих паттернов (`is None` и `not in`).

**Почему `_extract_detail` разворачивает и `str`, и `list`.** FastAPI возвращает:
- `{"detail": "Not authenticated"}` — `HTTPException(detail=str)`.
- `{"detail": [{"loc": ["body", "password"], "msg": "..."}]}` — Pydantic ValidationError (422).

`_extract_detail` собирает оба в строку. Иначе пользователь видел бы сырой JSON.

**Почему `if endpoint == "/health" and response.status_code == 503`.** `/health` возвращает 503 при недоступности любого сервиса — это **диагностический** ответ, не ошибка. Парсим как обычный JSON, чтобы UI показал `services.*.healthy: false` с деталями. Без этой ветки `raise_for_status()` бросил бы `HTTPError` и UI показал бы «API Error (503)» вместо детальной картины.

**Почему `if e.response.status_code == 401: _clear_session_state()`.** Токен истёк или невалиден. `_clear_session_state` сбрасывает `access_token`, `user`, удаляет ephemeral-ключи. Streamlit при следующем rerun покажет форму логина. Без — UI останется «залогиненным», но каждый запрос будет 401.

**Почему `url = f"{API_URL}{endpoint}"` с f-string.** `API_URL` — префикс, `endpoint` — путь. `/translate` + `http://localhost:8000` = `http://localhost:8000/translate`. Простой конкатенация, без `urljoin` (запутается на `//`).

**Почему `if "files" not in kwargs: headers.setdefault("Content-Type", "application/json")`.** Для `upload_file` `requests` сам ставит `multipart/form-data; boundary=...`. Если принудительно `application/json` — файл не отправится (500/400 от FastAPI). `setdefault` не перезаписывает существующий.

**Почему `quote(entry_id, safe='')`.** `entry_id` содержит `|` (SHA-256 формат: `<src>|<tgt>|<hash>`) и `/` может встретиться. `quote(..., safe='')` — кодирует всё, кроме `A-Za-z0-9_.-`. URL-safe. Без — FastAPI получит `id` с `|`, но маршрут `/admin/glossary/delete/{entry_id}` не матчится на `|` в URL (это «safe» в URL-спеке, но FastAPI может ругаться). `quote` — надёжно.

**Почему `logout()` = `_clear_session_state()`.** Не делает HTTP-запрос (JWT stateless, сервер не хранит сессии). Просто сбрасывает клиентское состояние. Правильно.

**Почему `translate`, `translate_multi`, `translate_batch`, `translate_ensemble` — отдельные функции.** Каждая — разный endpoint, разные Pydantic-модели. Универсальная `api_request("POST", "/translate", ...)` работает, но типизация/автодополнение хуже. Отдельные — понятнее для страниц.

**Почему `engine: str = "auto"` — не `Optional[TranslationEngine]`.** Streamlit не знает Pydantic-типы из backend. `str` — достаточно, FastAPI сам валидирует на своей стороне (422 при невалидном).

---

## 12.3. `streamlit_app/utils/auth_guard.py`

**Что это:** `require_login()` и `require_role(*roles)` — защита страниц. Вызываются в начале каждой страницы.

**Зачем это нужно:** Streamlit multi-page app: каждая страница — отдельный скрипт, выполняется независимо. Без guard пользователь может открыть `/pages/4_Glossary.py` напрямую (по URL) без логина/прав. `require_login` проверяет `st.session_state.user`, `require_role` — `user.role`.

Создайте `streamlit_app/utils/auth_guard.py`:

```python
# streamlit_app/utils/auth_guard.py
import streamlit as st


def current_user() -> dict:
    return st.session_state.get("user") or {}


def require_login() -> dict:
    user = current_user()
    if not user:
        st.error("🔒 Please log in to access this page.")
        st.page_link("app.py", label="→ Go to login", icon="🚀")
        st.stop()
    return user


def require_role(*allowed_roles: str) -> dict:
    user = require_login()
    role = user.get("role", "user")
    if role not in allowed_roles:
        st.error(
            f"🔒 This page requires one of the roles: "
            f"{', '.join(allowed_roles)}."
        )
        st.page_link("app.py", label="→ Go to dashboard", icon="🚀")
        st.stop()
    return user
```

### Разбор

**Почему `st.stop()`, а не `return`.** `st.stop()` останавливает выполнение скрипта. `return` — только из функции, но страница Streamlit — это top-level скрипт, `return` не работает вне функции. `st.stop()` — единственный способ.

**Почему `st.page_link("app.py", ...)`, а не `st.switch_page`.** `st.page_link` — ссылка на страницу. `st.switch_page` (новый API) — программный переход, но `page_link` работает в multi-page app как ссылка. Пользователь кликает → переходит.

**Почему `require_role` вызывает `require_login` первым.** Если не залогинен — `st.stop()` в `require_login` остановит. Если залогинен, но роль не подходит — `st.error` в `require_role`. Каскад: сначала «залогинься», потом «нет прав».

**Почему `user.get("role", "user")` с дефолтом.** `user` — dict из `session_state` (из `TokenResponse`). Если поля `role` нет (старый токен) — `"user"` (наименьшие права). Безопасный дефолт.

---

## 12.4. `streamlit_app/utils/helpers.py`

**Что это:** 3 утилиты — `entry_id`, `filter_value`, `short`.

Создайте `streamlit_app/utils/helpers.py`:

```python
# streamlit_app/utils/helpers.py

def entry_id(entry: dict) -> str:
    if entry.get("id"):
        return entry["id"]
    return (
        f"{entry['source_lang']}|{entry['target_lang']}|"
        f"{entry['en']}|{entry['tgt']}"
    )


def filter_value(label: str) -> str | None:
    return None if label == "All" else label


def short(text: str, n: int = 60) -> str:
    text = text.strip()
    return text if len(text) <= n else text[:n] + "..."
```

### Разбор

**Почему `entry_id` сначала `entry.get("id")`.** Бэкенд (Этап 7) возвращает `id` в `list_all` / `get_entries` — `<src>|<tgt>|<sha256>`. Если `id` есть — используем его. Если нет (старый бэкенд) — fallback на legacy-формат `<src>|<tgt>|<en>|<tgt>` (без sha). `quote(..., safe='')` в `delete_glossary` кодирует оба формата.

**Почему `filter_value("All") → None`.** Streamlit `selectbox(options=["All"] + list(LANGUAGES.keys()))`. «All» — sentinel. API принимает `None` = «не фильтровать». Функция конвертирует. Одна точка правды.

**Почему `short(text, n=60)`.** В expander'ах и selectbox-опциях длинные тексты не влезают. 60 символов + `...` — обозримо. `text.strip()` — обрезает пробелы до проверки длины.

---

## 12.5. `streamlit_app/app.py` — главная

**Что это:** страница логина/регистрации, dashboard с ссылками на 8 страниц, быстрый перевод.

Полный код — в `combined_streamlit_app_code.txt` (файл `streamlit_app/app.py`). Ключевые моменты:

**Структура:**

1. `st.set_page_config(...)` — layout, sidebar.
2. `st.session_state.setdefault("access_token", None)` и др. — инициализация.
3. `_validate_password(password)` — зеркалит серверную валидацию (≥ 8, цифра, заглавная, ≤ 72 байта) для мгновенной обратной связи.
4. Sidebar — `st.title("🚀 RUNG")`, логин/регистрация (2 таба) или «Logout».
5. Если не залогинен — `st.stop()` после welcome-текста.
6. Dashboard — `st.page_link` на 8 страниц (условно — только для admin/editor).
7. Quick translation — text area + selectbox'ы + кнопка, `quick_result` кэшируется в `session_state` по `quick_key`.

**Разбор ключевых решений:**

**Почему `quick_key = f"{text}|{src}|{tgt}|{engine}"` в `session_state`.** Если пользователь изменил текст/языки — старый `quick_result` не должен показываться. `quick_key` меняется → `quick_result = None`. Иначе после изменения textarea видели бы «старый» перевод к «новому» тексту.

**Почему `_validate_password` — дубликат серверной.** FastAPI вернёт 422 при невалидном пароле, но только после submit. `_validate_password` — до submit, мгновенно. UX: пользователь видит ошибку до отправки. Минус: политика дублируется в 2 местах (сервер + Streamlit). Если меняется — в обоих.

**Почему `role = st.session_state.user.get("role", "user")`.** `is_admin = role == "admin"`, `is_editor = role in ("admin", "editor")`. Условный рендеринг `st.page_link` — `4_Glossary.py` только для admin, `6_HITL.py` — для admin/editor. Скрытые ссылки — не защита (URL можно открыть вручную), но `require_role` в самих страницах защищает.

**Почему `quick_result` показывается через `st.success(...)` + `st.caption(...)`.** `st.success` — большой зелёный блок для перевода. `st.caption` — мелкий серый текст для `model_used` / `critic_score`. Минимальная визуализация.

**Почему `st.set_page_config(..., initial_sidebar_state="expanded")`.** Sidebar с логином/логаутом всегда виден. Пользователь сразу видит статус.

---

## 12.6. `streamlit_app/pages/1_Translation.py`

**Что это:** одиночный перевод (text, src, tgt, engine) + мультиязычный (один text, много target_langs). `reasoning_log` в expander'ах.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

**Одиночный перевод:**

- `st.session_state.single_key` — по `text|src|tgt|engine`. Сброс `single_result` при изменении.
- `translate(text, src, tgt, engine)` → `single_result`.
- `st.success` с переводом + `st.caption` с model/critic_score/engine.
- `st.expander("📋 Translation details")` — `st.json(summary)` без `reasoning_log`.
- `st.expander(f"🧠 Agent reasoning ({len(reasoning_log)} steps)")` — цикл по `reasoning_log`, для каждого шага `st.markdown` + `st.write` по полям + `st.divider()`.

**Мультиязычный:**

- `st.multiselect("Target languages", ...)` — default `["ru", "tg"]`.
- `translate_multi(text, src, target_langs, engine)` → `multi_result`.
- `for lang, translation in multi_result["translations"].items()` — `st.write(f"**{label}:** {translation}")` или `st.warning` для пустого.
- Expander с `st.json` деталей.

**Разбор:**

**Почему `reasoning_log` в expander'е.** Полный лог шагов агентов (Validator → Critic → Editor → ... → Supervisor final) — много строк. Expander сворачивает — не забивает основной UI.

**Почему `if value in (None, [], {}, ""): continue` в цикле по полям шага.** Поля могут быть `None` (например, `critic_score=None` для `baseline`), `[]` (suggestions), `""`. Не показываем пустые — фокус на значимых.

---

## 12.7. `streamlit_app/pages/2_Batch_Translation.py`

**Что это:** список текстов (по одному на строку) → `translate_batch`. Проверка `len(translations) == len(texts)`.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `texts_input.split("\n")` → `texts` (без пустых).
- `batch_key = f"{'|'.join(texts)}|{src}|{tgt}|{engine}"` — кэш результата в `session_state`.
- `translate_batch(texts, src, tgt, engine)` → `batch_result`.
- `if len(translations) != len(texts): st.error(...)` — защита от рассинхронизации.
- `pairs = list(zip(texts, translations))` — пары.
- `failures = sum(1 for _, trans in pairs if not trans or not trans.strip())`.
- Цикл `for i, (src, trans) in enumerate(pairs)` — `st.expander(label)`, `st.write(f"**Source:** {src}")` + `st.write(f"**Translation:** {trans}")`.

**Разбор:**

**Почему `zip` потенциально опасен.** `zip` обрезает до короткого. Если бэкенд вернул 2 перевода на 3 текста — `zip` дал бы 2 пары, ошибку не видно. **Поэтому** сначала `if len(translations) != len(texts): st.error(...)`. После проверки — `zip` безопасен.

**Почему `failures` считается, но не влияет на `st.success`.** `safe_perform_translation` (Этап 9.3) возвращает `(ok, result)`. Для упавших — `result["translation"] = ""`. `translations` содержит `""`. `failures` — количество пустых. UI показывает `st.warning(f"⚠️ Translated {len(pairs)} texts ({failures} empty)")` — честно.

**Почему `st.expander` для каждого текста.** 50 текстов в одном списке — нечитаемо. Expander сворачивает, видно первые 50 символов. Клик — раскрывается.

---

## 12.8. `streamlit_app/pages/3_Ensemble.py`

**Что это:** выбор моделей (из `/health.ensemble_models`), `translate_ensemble`. Disabled при неподдерживаемой паре.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `health = get_health() or {}` — читаем `ensemble_models` и `known_languages`.
- `available_models = health.get("ensemble_models") or _DEFAULT_MODELS` — fallback.
- `ollama_ready = set(health.get("known_languages") or [])` — 14 языков.
- `pair_supported = source_lang in ollama_ready and target_lang in ollama_ready`.
- `st.button("Run Ensemble Translation", disabled=not pair_supported)`.
- `translate_ensemble(text, src, tgt, models)` → `variants`, `best_translation`.
- Цикл `for variant in result.get("variants", [])` — expander с моделью + score.

**Разбор:**

**Почему `disabled=not pair_supported`.** Ensemble использует `KNOWN_LANGUAGES` (14). Для `udm`/`os`/`av`/`tk` — 400 от бэкенда. UI заранее предупреждает и блокирует.

**Почему `if health.get("ensemble_models") is None: st.info(...)`.** Если бэкенд старой версии (без `ensemble_models` в `/health`) — используем `_DEFAULT_MODELS`, но показываем info: «бэкенд не вернул список, использую дефолты». Прозрачность.

**Почему `default=available_models[:2]`.** По умолчанию 2 модели (не все 3) — быстрее. Пользователь может добавить третью.

---

## 12.9. `streamlit_app/pages/4_Glossary.py`

**Что это:** CRUD + CSV-импорт глоссария. `require_role("admin")`.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `require_role("admin")`.
- Add: form с `en`, `tgt`, `source_lang`, `target_lang` → `add_glossary`.
- Import CSV: `st.file_uploader(type="csv")` → `import_glossary_csv(file)`.
- List: `list_glossary(source_lang, target_lang, limit, offset)` → `st.dataframe`.
- Delete: `st.selectbox` из `entry_id(e)` → `delete_glossary(selected_id)`.
- Confirm: `st.checkbox("I confirm...")` перед delete.

**Разбор:**

**Почему `st.session_state.glossary_data` кэширует.** `list_glossary` — HTTP-запрос. При каждом rerun (любое изменение виджета) — не должен перезапрашивать. `glossary_filter_key = f"{filter_src}|{filter_tgt}|{limit}|{offset}"` — сброс при изменении фильтров. Кнопка `🔄 Refresh List` — ручной сброс.

**Почему `options=[entry_id(e) for e in entries]`.** Каждая опция — `id` (UUID-строка или `<src>|<tgt>|<sha>`). `format_func=lambda x: labels.get(x, x)` — отображение через словарь. Нельзя передать `GlossaryEntry` в `selectbox` (не hashable).

**Почему `st.checkbox(..., key=f"glossary_delete_confirm_{selected_id}")`.** `key` уникален для каждого `selected_id`. Если выбрать другой entry — checkbox сбросится (новый `key`). Без уникального `key` — checkbox «залипал» бы.

---

## 12.10. `streamlit_app/pages/5_TM.py`

**Что это:** то же, что 4_Glossary.py, но для TM. `require_role("admin")`.

Код — в `combined_streamlit_app_code.txt`. Отличия от 4_Glossary:

- `list_tm`, `add_tm`, `delete_tm`, `import_tm_csv` — вместо glossary.
- `st.text_area("Source text", height=80)` вместо `text_input` — TM-пары длиннее.
- `short(e['en'], 60)` в `labels` — TM-тексты могут быть длинными.
- `st.session_state.tm_data`, `tm_filter_key`.

**Разбор:** см. 4_Glossary.py.

---

## 12.11. `streamlit_app/pages/6_HITL.py`

**Что это:** очередь `pending` примеров, confirm/reject, ручное сохранение корректировки. `require_role("admin", "editor")`.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `require_role("admin", "editor")`.
- Stats: `get_uncertain_stats()` → `st.metric` × 3 (pending, confirmed, rejected).
- Manual save: form с `source_text`, `source_lang`, `target_lang`, `corrected_translation` → `save_correction`.
- Queue: `get_uncertain(src, tgt, limit, offset)` → `examples`.
- Для каждого: `st.expander(f"ID: {id[:8]}… | Score: {score}")` + text_area для corrected + 2 кнопки (confirm, reject).

**Разбор:**

**Почему `format_score(score)` — функция.** `score` может быть `None` (Critic упал). `f"{score:.2f}"` упал бы на `None`. `"N/A" if score is None else f"{score:.2f}"` — правильно.

**Почему `text_area(value=example["machine_translation"], key=f"corrected_{example_id}")`.** Поле **предзаполнено** машинным переводом. Редактор правит, а не пишет с нуля. `key` уникален — каждое поле привязано к своему примеру.

**Почему `if not corrected.strip(): st.warning(...)` перед `confirm_uncertain`.** Пустая корректировка — бессмысленна (`Pydantic` на бэкенде вернёт 422). Проверяем на клиенте — быстрее.

**Почему `result = confirm_uncertain(...)` + `if result: st.success(...) + st.session_state.uncertain_data = None + st.rerun()`.** `confirm_uncertain` возвращает `None` при ошибке (HTTP), dict при успехе. При успехе — сброс кэша очереди + rerun (очередь обновится).

---

## 12.12. `streamlit_app/pages/7_Evaluation.py`

**Что это:** upload JSON или ручной ввод, метрики BLEU/chrF++/BERTScore/METEOR, `N/A` для отсутствующих.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `require_login()`.
- Таб 1 (Upload JSON): `st.file_uploader(type="json")` → `run_evaluation(sources, hypotheses, references)`.
- Таб 2 (Manual): 3 text_area (sources/hypotheses/references) → `_validate_lengths` → `run_evaluation`.
- `render_metrics(result)` — 4 `st.metric` + caption «Not computed: ...» для `None`.

**Разбор:**

**Почему `_validate_lengths` — до `run_evaluation`.** Бэкенд вернёт 400 при несовпадении длин. Клиент валидирует заранее — быстрее фидбек.

**Почему `_METRIC_KEYS` — tuple.** `("BLEU", "bleu", ".4f")`, `("chrF++", "chrf_plus_plus", ".2f")`, `("BERTScore", "bertscore", ".4f")`, `("METEOR", "meteor", ".4f")`. Формат (`spec`) — разный: BLEU/BERTScore/METEOR — 4 знака, chrF++ — 2 (обычно 0–100, 2 знака).

**Почему `_format_metric(value, spec)` — `"N/A"` при `None`.** Metric может быть `None`, если бэкенд не смог вычислить (например, `evaluate` не установлен). `f"{None:.4f}"` — `TypeError`. `"N/A"` — честно.

**Почему `render_metrics` + `missing`.** `st.metric(label, "N/A")` — серое «N/A». `st.caption(f"Not computed: {', '.join(missing)}")` — пояснение, что метрика не посчиталась (не ошибка).

---

## 12.13. `streamlit_app/pages/8_System_Health.py`

**Что это:** статусы сервисов, языки, статистика кэша, очистка кэша. `require_role("admin")`.

Код — в `combined_streamlit_app_code.txt`. Ключевое:

- `require_role("admin")`.
- `get_health()` → `services` → `st.columns` × N, каждая — `st.metric` healthy/unhealthy + error caption.
- Supported languages → `st.dataframe` (code, name, ollama_ready ✓).
- Cache stats: `get_cache_stats()` → `st.metric` × 2.
- Clear cache: selectbox src/tgt/eval_mode + checkbox + button → `clear_cache(...)`.
- `_EVAL_MODES = ["All", "baseline", "glossary", "tm", "tm_hybrid", "agents_only", "tm_hybrid_critic", "tm_hybrid_editor", "full"]` — hardcoded (синхронно с `pipeline._MODE_CONFIG`).

**Разбор:**

**Почему `_EVAL_MODES` hardcoded в Streamlit, а не из API.** Backend не отдаёт список режимов (нет эндпоинта). Streamlit копирует. Если добавите 9-й режим в `_MODE_CONFIG` — не забудьте и здесь. Комментарий в коде фиксирует.

**Почему `if src_lang == "All" and tgt_lang == "All" and eval_mode == "All": st.warning("This will remove every cache entry...")`.** Опасная операция (полная очистка кэша). Предупреждение — до checkbox. Пользователь осознанно нажимает.

**Почему `confirm = st.checkbox("I understand this action cannot be undone", key="clear_cache_confirm")` + `disabled=not confirm`.** Защита от случайного клика. Двухшаговое подтверждение: checkbox + кнопка.

**Почему `filter_value(src_lang)` → `None` для «All».** `clear_cache` с `None` — «не фильтровать» (Этап 9.2). Соответствует семантике API.

---

## 12.14. `streamlit_app/requirements.txt`

```
streamlit>=1.30
requests>=2.31
```

> **Версии.** `streamlit 1.30+` — `st.page_link` (появился в 1.29), `st.container(border=True)` (1.29). Более старые — не поддерживают часть API. `requests` — HTTP-клиент.

Установка:

```powershell
python -m pip install -r streamlit_app/requirements.txt
```

Запуск:

```powershell
streamlit run streamlit_app/app.py --server.port 8501
```

**Что вы должны увидеть:**

```
  You can now view your Streamlit app in your browser.
  Local URL: http://localhost:8501
  Network URL: http://192.168.x.x:8501
```

Откройте `http://localhost:8501`. Должна открыться страница логина.

> **`API_URL`.** По умолчанию `http://localhost:8000` — работает, если `app` (бэкенд) запущен локально на `8000` (через `docker compose up` — порт 8000 проброшен). Если бэкенд за nginx — `http://localhost` (порт 80). Настройте в `.streamlit/secrets.toml`:
> ```toml
> API_URL = "http://localhost:8000"
> ```

---

## 12.15. `infrastructure/docker-compose.yml` — сервис `nginx`

Откройте `infrastructure/docker-compose.yml` (Этап 11: `db`, `redis`, `qdrant`, `ollama`, `ollama-init`, `app`, `worker`, `beat`). Добавьте **после `beat`**:

```yaml
  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf:ro
      - ../static:/usr/share/nginx/html/static:ro
    depends_on:
      app:
        condition: service_started
    restart: unless-stopped
    networks:
      - rung_internal
```

### Разбор

**Почему `nginx:alpine`.** ~10 МБ, официальный образ. `alpine` — минимальный.

**Почему `ports: "80:80"`.** Внешний порт 80 (HTTP) → внутренний 80. Пользователь открывает `http://localhost/` — nginx проксирует на `app:8000`. Без nginx — `http://localhost:8000` (порт `app` проброшен).

**Почему `volumes: ./nginx.conf:/etc/nginx/conf.d/default.conf:ro`.** Монтируем наш конфиг в дефолтный конфиг nginx. `:ro` — read-only. Nginx читает конфиг при старте; изменение файла требует `docker compose restart nginx`.

**Почему `../static:/usr/share/nginx/html/static:ro`.** Если в проекте есть `static/` (CSS/JS/изображения) — nginx отдаёт их напрямую (без проксирования на `app`). Быстрее для статики. Если `static/` нет — папка создаётся пустой, `location /static/` вернёт 404, но не сломает остальное.

**Почему `depends_on: app: service_started`, а не `service_healthy`.** Nginx не требует, чтобы `app` был готов принимать запросы — он проксирует, и если `app` не готов, вернёт 502. `service_started` — «контейнер создан», этого достаточно. `service_healthy` добавил бы 180 с ожидания (`start_period: 180s` у `app`) — nginx простаивал бы.

**Почему `restart: unless-stopped`.** Nginx стабилен, но при падении — рестарт. `unless-stopped` — стандарт.

---

## 12.16. `infrastructure/nginx.conf`

Полный код — в `combined_config_files.txt` (файл `infrastructure/nginx.conf`). Ключевое:

```nginx
map $http_upgrade $connection_upgrade {
    default upgrade;
    ''      close;
}

server {
    listen 80;
    server_name localhost;
    server_tokens off;

    client_max_body_size 100M;

    gzip on;
    gzip_types text/plain application/json application/javascript text/css application/xml;
    gzip_min_length 1024;
    gzip_comp_level 5;

    add_header X-Frame-Options "SAMEORIGIN" always;
    add_header X-Content-Type-Options "nosniff" always;
    add_header X-XSS-Protection "1; mode=block" always;
    add_header Referrer-Policy "strict-origin-when-cross-origin" always;

    location / {
        proxy_pass http://app:8000;

        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection $connection_upgrade;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        proxy_connect_timeout 60s;
        proxy_send_timeout 600s;
        proxy_read_timeout 600s;

        proxy_buffering off;

        proxy_set_header X-Forwarded-Host $host;
        proxy_set_header X-Forwarded-Port $server_port;

        proxy_ignore_client_abort on;
    }

    location /static/ {
        alias /usr/share/nginx/html/static/;
        expires 1y;
        add_header Cache-Control "public, immutable";
        access_log off;
    }

    location /health {
        proxy_pass http://app:8000/health;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_connect_timeout 5s;
        proxy_read_timeout 5s;
        access_log off;
    }
}
```

### Разбор ключевых решений

**Почему `map $http_upgrade $connection_upgrade`.** Для WebSocket (если появятся): nginx должен передать `Connection: upgrade`. `$http_upgrade` — заголовок от клиента. `map` — если `upgrade` → `connection_upgrade=upgrade`, иначе `close`. Сейчас WebSocket не используется, но `map` — стандарт.

**Почему `proxy_read_timeout 600s`.** `httpx.AsyncClient(timeout=300.0)` в `http_client.py` — 300 с на запрос. `proxy_read_timeout` должен быть ≥, иначе nginx оборвёт соединение раньше. 600 с = 2× запас. Для `batch` на 50 текстов (семафор 4, ~50 × 2 с / 4 = 25 с) — 600 с с запасом. Для `full` режима с агентами — 1 запрос может быть 60–120 с. 600 с — безопасно.

**Почему `proxy_buffering off`.** Nginx по умолчанию буферизует ответ бэкенда (накапливает, потом отдаёт). Для длинных стриминговых ответов это плохо — клиент ждёт, пока накопится буфер. `off` — отдаёт по мере поступления. Для Streamlit/SSE — нужно. Сейчас стриминга нет, но `off` — стандарт для API.

**Почему `client_max_body_size 100M`.** `POST /admin/glossary/import-csv` с CSV на 10000 строк — несколько МБ. `POST /admin/tm/import-csv` — тоже. 100 МБ — с запасом. Дефолт nginx (1 МБ) обрезал бы.

**Почему `server_tokens off`.** Скрывает версию nginx в заголовке `Server:`. Безопасность (меньше информации для атакующего).

**Почему security headers `X-Frame-Options`, `X-Content-Type-Options`, `X-XSS-Protection`, `Referrer-Policy`.** Стандартные:
- `X-Frame-Options: SAMEORIGIN` — запрет `<iframe>` с других доменов (clickjacking).
- `X-Content-Type-Options: nosniff` — браузер не «угадывает» MIME, использует `Content-Type`.
- `X-XSS-Protection: 1; mode=block` — старый XSS-фильтр (deprecated, но безвреден).
- `Referrer-Policy: strict-origin-when-cross-origin` — контроль `Referer`.

**Почему `location /static/` с `expires 1y`.** Статика (CSS/JS) не меняется — кэш на 1 год. `Cache-Control: public, immutable` — браузер не перепроверяет. `access_log off` — не логируем статику (шум).

**Почему отдельный `location /health` с 5 с.** `/health` — быстрый (SELECT 1 + probe). 600 с там излишни. 5 с — «сервис не отвечает». Docker healthcheck на `app` использует `/health` (5 с timeout) — согласовано. `access_log off` — healthcheck каждые 15 с не забивает логи.

**Почему `X-Forwarded-For` / `X-Forwarded-Proto` / `X-Forwarded-Host` / `X-Forwarded-Port`.** FastAPI за nginx видит IP nginx (внутренний), не клиента. `X-Forwarded-*` — стандарт, по которому FastAPI (если настроен `ProxyHeadersMiddleware`) восстанавливает оригинальный IP/схему. Сейчас не используется (FastAPI не читает эти заголовки без middleware), но `add_header` — стандарт.

**Почему `proxy_ignore_client_abort on`.** Если клиент закрыл соединение (Streamlit перезагрузился), nginx **продолжит** передачу ответа от `app`. Без `on` — nginx оборвёт, `app` получит `BrokenPipeError`. `on` — дожидаемся полного ответа, потом закрываем. Полезно для длинных переводов: пользователь не потеряет результат, даже если обновит страницу.

---

## 12.17. `README.md`

Полный `README.md` — в `combined_config_files.txt` (или создать заново). Структура:

```markdown
# RUNG — Retrieval-Augmented Neural Generation

**Платформа машинного перевода с гибридным RAG и мультиагентной системой.**

## 📋 О проекте

[2-3 абзаца: что делает, для кого, ключевые компоненты]

## ✨ Возможности

- Мультиагентный pipeline: Validator → Critic → Editor (Supervisor с adaptive budget и rollback)
- Гибридный RAG: BM25 + Qdrant (cosine) + RRF
- 8 ablation-режимов: baseline, glossary, tm, tm_hybrid, agents_only, tm_hybrid_critic, tm_hybrid_editor, full
- Два движка LLM: Ollama (локально) и HuggingFace Transformers (fallback)
- Human-in-the-Loop: очередь uncertain_examples, confirm/reject
- Кэш переводов в БД (6 полей ключа, счётчик hits)
- Celery: переиндексация, batch-перевод, очистка кэша
- JWT auth с ролями user / editor / admin
- Streamlit UI (8 страниц)

## 🏗️ Архитектура

```
[ASCII-диаграмма или ссылка на скриншот]
```

Ключевые файлы:
- `backend/services/pipeline.py` — центральный `perform_translation`, 8 режимов
- `backend/agents/` — Supervisor + Validator + Critic + Editor
- `backend/retrieval/` — BM25 + Qdrant + RRF
- `backend/hitl/` — ActiveLearningManager, FeedbackHandler
- `backend/worker/` — Celery app + 6 задач
- `streamlit_app/` — UI (8 страниц)
- `infrastructure/` — Docker + Nginx

## 🚀 Быстрый старт

### Требования
- Docker 24+
- Docker Compose v2
- 16 ГБ RAM (Ollama: qwen2.5:14b + qwen2.5:7b)
- GPU (опционально, для HF NLLB)

### Запуск

```bash
git clone <repo>
cd rung
cp infrastructure/.env.example infrastructure/.env
# Отредактировать .env (HF_TOKEN, JWT_SECRET_KEY, POSTGRES_PASSWORD)
docker compose -f infrastructure/docker-compose.yml up -d
```

Через 5-10 минут (скачивание моделей):
- API: http://localhost/docs
- Health: http://localhost/health
- Streamlit: `streamlit run streamlit_app/app.py --server.port 8501` (отдельно, локально)

### Первый админ

```bash
python -m poetry run python scripts/import/create_admin.py \
    --email admin@rung.local \
    --username admin \
    --password Admin1234
```

### Загрузка данных

```bash
python -m poetry run python scripts/import/add_glossary.py
python -m poetry run python scripts/import/add_tm.py
```

Или через `POST /admin/glossary/import-csv` с CSV.

## 📊 Данные

| Пара | Glossary | TM |
|---|---|---|
| ar_ru | 505 | 2500 |
| ru_tt | 510 | 1100 |
| tg_fa | 500 | 1000 |
| ru_ba | 505 | 1000 |
| **Итого** | **2020** | **5600** |

Источники — см. `data/import/README.md`.

## 🧪 Тесты

```bash
python -m poetry run pytest tests/ -v
```

Покрытие ≥ 70% (`--cov=backend`).

## 🎯 Ablation-режимы

| Режим | Глоссарий | TM | Critic | Editor |
|---|---|---|---|---|
| baseline | ✗ | — | ✗ | ✗ |
| glossary | ✓ | — | ✗ | ✗ |
| tm | ✓ | BM25 | ✗ | ✗ |
| tm_hybrid | ✓ | hybrid | ✗ | ✗ |
| agents_only | ✗ | — | ✓ | ✓ |
| tm_hybrid_critic | ✓ | hybrid | ✓ | ✗ |
| tm_hybrid_editor | ✓ | hybrid | ✗ | ✓ |
| full | ✓ | hybrid | ✓ | ✓ |

Выбор — через `EVAL_MODE` в `.env` или `eval_mode` в `/translate`.

## 🖥️ Streamlit UI

Запуск отдельно (не в Docker):
```bash
pip install -r streamlit_app/requirements.txt
streamlit run streamlit_app/app.py --server.port 8501
```

8 страниц: Translation, Batch, Ensemble, Glossary, TM, HITL, Evaluation, System Health.

## ⚠️ Известные ограничения

- **Streamlit не в Docker Compose** — dev-инструмент, запускается локально.
- **`beat_schedule` не настроен** — `beat` контейнер работает вхолостую. `clear_old_cache` — только вручную.
- **`batch_translate` не partial-fail-safe** — один упавший текст перезапускает весь batch.
- **`--concurrency=1`** — 1 задача одновременно.
- **`index_translation_memory` — полная переиндексация** (drop + create Qdrant). Для больших БД нужен incremental.
- **`TranslationCache` без TTL** — старые записи чистятся только `clear_old_cache`.

## 📚 Документация

- `docs/week-*.md` — отчёты по этапам
- `data/import/README.md` — источники данных
- `scripts/README.md` — скрипты (eval, import, human_eval)

## 👥 Авторы

[Ваше имя]
```

### Разбор

**Почему `README.md` — критично.** Первое, что видит проверяющий. `docker compose up -d` должен работать после `cp .env.example .env`. Без README — проект «непонятный».

**Почему «Известные ограничения» — раздел.** Честность: `beat_schedule` не настроен, `batch_translate` не partial-fail-safe. Если проверяющий найдёт это сам — плохо. Если вы указали — показывает, что понимаете архитектуру.

**Почему «Быстрый старт» в начале.** Проверяющий (и вы через месяц) не должен искать инструкцию по запуску. Первая команда — `docker compose up`.

**Почему `docs/week-*.md`.** Отчёт по этапам — требование ТЗ. 12 файлов (по одному на этап). Что сделано, что не получилось, что перенесено.

---

## 12.18. `tests/integration/test_end_to_end.py`

**Что это:** полный flow: register → login → translate → cache hit → HITL confirm → translate uses new TM.

Код:

```python
# tests/integration/test_end_to_end.py
"""
End-to-end test: register, login, translate, cache hit,
HITL confirm, translate picks up new TM entry.
"""
import pytest
from fastapi.testclient import TestClient

from backend.api.main import app


@pytest.fixture
def client():
    return TestClient(app)


def test_full_user_journey(client):
    # 1. Register
    reg = client.post("/auth/register", json={
        "email": "e2e@test.local",
        "username": "e2e_user",
        "password": "TestPass1234",
    })
    assert reg.status_code == 201
    token = reg.json()["access_token"]
    headers = {"Authorization": f"Bearer {token}"}

    # 2. Login (same credentials)
    login = client.post("/auth/login", json={
        "email": "e2e@test.local",
        "password": "TestPass1234",
    })
    assert login.status_code == 200

    # 3. Translate
    text = "The quick brown fox jumps over the lazy dog"
    r1 = client.post("/translate", json={
        "text": text, "source_lang": "en", "target_lang": "ru",
    }, headers=headers)
    assert r1.status_code == 200
    body1 = r1.json()
    assert body1["translation"]
    assert body1["translator_used"] in ("ollama", "huggingface")

    # 4. Second translate: should hit cache
    r2 = client.post("/translate", json={
        "text": text, "source_lang": "en", "target_lang": "ru",
    }, headers=headers)
    assert r2.status_code == 200
    body2 = r2.json()
    assert body2["translator_used"] == "cache"
    assert body2["translation"] == body1["translation"]

    # 5. If Critic scored low, example is in HITL queue
    # (depends on actual LLM; may or may not be present)
    uncertain = client.get("/hitl/uncertain", headers=headers)
    # 200 if admin/editor; 403 for regular user — depends on role
    # For E2E, we assume the test user is admin (created separately)
    # OR we only check the queue is reachable

    # 6. Verify TM count grows after manual add
    # (already covered by unit tests; skip here)
```

### Разбор

**Почему E2E — не «всё в одном».** Полный flow (register → ... → confirm → translate uses TM) требует admin/editor роли и реального LLM. В CI без Ollama — не пройдёт. **E2E в референсе** — простой (register → login → translate → cache hit). HITL/confirm — в unit-тестах Этапа 10 (моки). Роль `admin` для E2E — через `monkeypatch` или предварительный `create_admin.py`.

**Почему `client = TestClient(app)` на module-level в референсе (хотя в Этапе 4 — fixture).** Оба варианта работают. Fixture — чище (изоляция). Module-level — быстрее. Для E2E — fixture (потому что нужна изоляция, тест долгий).

**Почему `if body2["translator_used"] == "cache"` — правильная проверка.** `perform_translation` (Этап 9.3) для кэш-hit ставит `translator="cache"`. Это единственный способ отличить кэш-hit от нового перевода. `translation` одинаковый в обоих случаях, `translator_used` — разный.

**Почему `reg.status_code == 201` (не 200).** `/auth/register` (Этап 4.3) — `status_code=status.HTTP_201_CREATED`. 201 — «создан ресурс».

**Почему без реального LLM E2E не пройдёт.** `/translate` вызывает `perform_translation` → `translator_manager.get_translator_for_engine` → Ollama (или HF). Без Ollama-контейнера — `ConnectError` → 502. Для CI: `monkeypatch` на `translator_manager` (mock) или `TRANSLATION_ENGINE=huggingface` (HF на CPU, медленно но работает). В референсе E2E — **с** реальным Ollama (запущен `docker compose up`). В CI — mock.

---

## 12.19. Бонусные скрипты (опционально, +10%)

Если хотите +10% — добавьте eval-скрипты:

### `scripts/eval/mode_labels.py`

Реестр 8 режимов (`MODE_ORDER`, `MODE_LABELS`). Единый источник правды для `run_eval.py`, `paired_bootstrap.py`, `report_tables.py`. Полный код — в `combined_python_code.txt`.

### `scripts/eval/run_eval.py`

Прогон 8 режимов на датасете (`eval_datasets/<pair>_<size>.json`), сохранение `<mode>.json` с гипотезами и метриками. `strict_engine=True` — eval не должен молча деградировать в HF. Полный код — в `combined_python_code.txt`.

### `scripts/eval/paired_bootstrap.py`

Парный bootstrap для статзначимости (`chrf_plus_plus`, `bleu`, `meteor`). `--all-pairs` — все пары режимов. WMT-пороги (`2.8` chrF, `3.0` BLEU). Полный код — в `combined_python_code.txt`.

### `scripts/eval/report_tables.py`

Публикационные таблицы: `summary.md/.tex`, `by_domain`, `failure_rates`, `significance_<metric>`. Читает `<mode>.json` + `paired_bootstrap.json`. Полный код — в `combined_python_code.txt`.

### `scripts/eval/check_tm_leakage.py`

Детектор утечки TM↔test (chrF++ пороги). `TM_LEAK_DROP_DB=1` — удалить утёкшие. Полный код — в `combined_python_code.txt`.

### `scripts/human_eval/extract_samples.py` и `analyze.py`

Blinded annotation sheet + Krippendorff's α. Полный код — в `combined_python_code.txt`.

**Объём.** 6 файлов, ~1500 строк. Для +10% — опционально. Для сдачи без бонусов — не нужно. Если добавляете — `README.md` упомяните в разделе «Бонусы».

---

## 12.20. Финальная проверка

### Запуск полного стека

```powershell
cd D:\RUNG\infrastructure
docker compose down
docker compose up -d --build
```

**Что вы должны увидеть:**

```
[+] Running 9/9
 ✔ Container infrastructure-db-1        Healthy
 ✔ Container infrastructure-redis-1     Healthy
 ✔ Container infrastructure-qdrant-1    Healthy
 ✔ Container infrastructure-ollama-1    Healthy
 ✔ Container infrastructure-ollama-init-1 Exited
 ✔ Container infrastructure-app-1       Started
 ✔ Container infrastructure-worker-1    Started
 ✔ Container infrastructure-beat-1      Started
 ✔ Container infrastructure-nginx-1     Started
```

`docker compose ps` — все `Up`/`healthy`. `ollama-init` — `Exited (0)`.

### Проверка nginx

```powershell
curl http://localhost/health
```

**Что вы должны увидеть:**

```json
{"status":"healthy","services":{"database":{"healthy":true},...}}
```

`curl http://localhost/docs` — Swagger UI (через nginx → `app:8000`).

`curl http://localhost/api/...` — **404** (nginx не имеет `location /api/`, всё через `/`). Все пути — прямые (`/translate`, `/health`, ...).

### Проверка Streamlit

```powershell
streamlit run streamlit_app/app.py --server.port 8501
```

Откройте `http://localhost:8501`. Логин `admin@rung.local` / `Admin1234`. Dashboard с 8 страницами.

### Проверка end-to-end

1. **Translation** — введите текст, выберите языки, «Translate» → перевод.
2. **Batch** — 3 текста → 3 перевода.
3. **Ensemble** — выберите 2 модели → варианты + best.
4. **Glossary** — добавьте термин → отображается в списке.
5. **TM** — добавьте пару → отображается.
6. **HITL** — если есть pending → confirm → статус меняется.
7. **Evaluation** — 2-3 пары (hypothesis, reference) → метрики.
8. **System Health** — статусы сервисов, кэш-статистика, очистка.

### Тесты

```powershell
python -m poetry run pytest tests/ -v
```

**Что вы должны увидеть:** все тесты зелёные, покрытие ≥ 70%.

Финальный чек-лист:

- [ ] `docker compose ps` — все `healthy`/`Up`, 9 сервисов.
- [ ] `curl http://localhost/health` — 200 через nginx.
- [ ] `curl http://localhost/docs` — Swagger.
- [ ] Streamlit на `:8501` — работает.
- [ ] 8 страниц Streamlit открываются.
- [ ] Логин/регистрация работают.
- [ ] `/translate` через UI — перевод.
- [ ] HITL: очередь видна, confirm/reject работают.
- [ ] `pytest tests/ -v` — зелёный, покрытие ≥ 70%.
- [ ] `README.md` — quick start, архитектура, данные, ограничения.
- [ ] `data/import/README.md` — источники.
- [ ] `docs/week-1.md` ... `docs/week-12.md` — отчёты.

### Git

```powershell
git add -A
git commit -m "feat(week-12): complete Streamlit UI, nginx, README, e2e test"
git push -u origin week-12
```

PR `week-12 → main`, merge.

---

## Что дальше (после сдачи)

**Защита проекта** (по правилам из Этапа 12):

1. **Живой прогон** через `/translate` с `reasoning_log` — Supervisor, Validator, Critic, Editor.
2. **Выключить Ollama** (`docker compose stop ollama`) → `/translate` → fallback на HF (`engine_used: "huggingface"`). При `strict_engine=True` — `EngineFallbackTriggeredError` (503).
3. **HITL-цикл:** `critic_score < 7` → запись в `/hitl/uncertain` → confirm → TM пополняется → повторный `/translate` использует новую TM-запись.
4. **Вопросы:**
   - Почему RRF (`1/(k+rank+1)`)?
   - Что если Qdrant недоступен? (BM25-only, warning)
   - Где Supervisor может зациклиться? (`_adaptive_budget`, rollback, `no_change`)
   - Почему кэш хранит `engine_used` и `eval_mode`? (6-полевой ключ, разделение прогонов)
   - Зачем adaptive budget вместо `max_iterations=3`? (экономия: 3 итерации только когда score < 4)
   - Почему rollback, а не просто `max_iterations`? (Editor может ухудшить — откат)
   - Почему `uuid5`, а не `uuid4` в Qdrant? (идемпотентный upsert)

**Бонусы** (если реализовали):
- `paired_bootstrap.py` — значимость различий.
- `report_tables.py` — публикационные таблицы.
- `check_tm_leakage.py` — утечка TM↔test.
- `human_eval/` — Krippendorff's α.
- CI/CD, Prometheus, WebSocket, rate limiting.

---

## Итог курса

**12 этапов, ~14 недель.** От «установить Python» до production-стека:
- Мультиагентный pipeline (Supervisor + Validator + Critic + Editor).
- Гибридный RAG (BM25 + Qdrant + RRF).
- 8 ablation-режимов для eval.
- Два LLM-движка (Ollama + HF fallback).
- HITL + Active Learning.
- Celery (6 задач, persistent loop, shutdown).
- JWT auth (user/editor/admin).
- Streamlit UI (8 страниц).
- Docker Compose (9 сервисов) + Nginx.
- Тесты (≥ 70% покрытия).

**README** — единственная точка входа для проверяющего. **Известные ограничения** — честность. **docs/week-*.md** — история решений.